# Adım 6: Makine Öğrenmesi — Çoklu Model Karşılaştırma + MLflow

**Hedef:** `avg_temp_c` (günlük ortalama sıcaklık) tahmini — **Regresyon Problemi**

| # | Model |
|---|-------|
| 1 | Linear Regression |
| 2 | Decision Tree Regressor |
| 3 | Random Forest Regressor |
| 4 | GBT Regressor |
| 5 | Generalized Linear Regression (GLR) |

**Metrikler:** RMSE · MAE · R²  
**Araç:** Spark MLlib + MLflow  
**Kaynak:** `/delta-lake/gold/ml_features`

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import mlflow
import mlflow.spark
import warnings
warnings.filterwarnings('ignore')

spark = (
    SparkSession.builder
    .appName('ClimateML')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

# MLflow: deneyleri yerel dosya sisteminde sakla
mlflow.set_tracking_uri('file:///home/jovyan/work/mlflow')
mlflow.set_experiment('climate_temperature_regression')

print(f'Spark surumu: {spark.version}')
print(f'MLflow surumu: {mlflow.__version__}')
print('Spark Session ve MLflow hazir ✅')

2026/05/10 21:42:00 INFO mlflow.tracking.fluent: Experiment with name 'climate_temperature_regression' does not exist. Creating a new experiment.


Spark surumu: 3.5.0
MLflow surumu: 3.12.0
Spark Session ve MLflow hazir ✅


In [2]:
ML_FEATURES_PATH = '/home/jovyan/work/delta-lake/gold/ml_features'
df = spark.read.format('delta').load(ML_FEATURES_PATH)

print(f'Toplam satir: {df.count():,}')
print(f'Sutun sayisi: {len(df.columns)}')
print('\nMevcut sutunlar:')
print(df.columns)
print('\nOrnek veri:')
df.show(3, truncate=False)

Toplam satir: 10,000
Sutun sayisi: 23

Mevcut sutunlar:
['measurement_date', 'city_name', 'station_id', 'season', 'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'year', 'month', 'day', 'temp_range_c', 'is_freezing', 'is_hot_day', 'precipitation_category', 'is_rainy', 'day_of_year', 'is_weekend', 'season_numeric', 'temp_lag_1', 'temp_diff_from_yesterday', 'temp_rolling_avg_7d', 'is_temp_anomaly']

Ornek veri:
+----------------+---------+----------+------+----------+----------+----------+----------------+----+-----+---+------------------+-----------+----------+----------------------+--------+-----------+----------+--------------+----------+------------------------+-------------------+---------------+
|measurement_date|city_name|station_id|season|avg_temp_c|min_temp_c|max_temp_c|precipitation_mm|year|month|day|temp_range_c      |is_freezing|is_hot_day|precipitation_category|is_rainy|day_of_year|is_weekend|season_numeric|temp_lag_1|temp_diff_from_yesterday|temp_rolling_avg_7

In [3]:
import pandas as pd

total = df.count()
print('=== EGITIM ONCESI NULL ANALIZI ===\n')

null_info = []
for c in df.columns:
    null_count = df.filter(col(c).isNull()).count()
    null_pct = round((null_count / total) * 100, 1)
    null_info.append((c, null_count, null_pct))

null_pd = pd.DataFrame(null_info, columns=['Sutun', 'Null Sayisi', 'Null %'])
null_pd = null_pd[null_pd['Null Sayisi'] > 0].sort_values('Null %', ascending=False)
print(null_pd.to_string(index=False))
print('\nNot: Null degerler Imputer ile medyan ile doldurulacak.')

=== EGITIM ONCESI NULL ANALIZI ===

                   Sutun  Null Sayisi  Null %
            temp_range_c         3830    38.3
        precipitation_mm         3610    36.1
              min_temp_c         2562    25.6
              max_temp_c         1856    18.6
              temp_lag_1            1     0.0
temp_diff_from_yesterday            1     0.0

Not: Null degerler Imputer ile medyan ile doldurulacak.


## Preprocessing Pipeline

Spark MLlib modelleri ham veriyi doğrudan alamaz. 3 adımlı bir pipeline kuruyoruz:

1. **Imputer** — `temp_lag_1`, `temp_rolling_avg_7d`, `temp_diff_from_yesterday` sütunlarındaki null değerleri medyan ile doldurur
2. **StringIndexer** — `precipitation_category` (None/Light/Moderate/Heavy) gibi metin değerini 0,1,2,3 sayılarına çevirir
3. **VectorAssembler** — Tüm sayısal feature sütunlarını tek bir `features` vektöründe birleştirir (modeller bunu ister)

In [4]:
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
    GeneralizedLinearRegression
)
from pyspark.ml.evaluation import RegressionEvaluator

# Hedef degisken
TARGET = 'avg_temp_c'

# Null icermeyen, dogrudan kullanilabilir feature'lar
STABLE_FEATURES = [
    'month',           # Ay (1-12) — mevsimsel dongü
    'year',            # Yil — uzun vadeli trend
    'day_of_year',     # Yilin günü (1-366) — sürekli mevsimsel sinyal
    'season_numeric',  # Sezon kodu (1=Kis, 2=Ilkbahar, 3=Yaz, 4=Sonbahar)
    'is_weekend',      # Hafta sonu mu? (0/1)
    'is_rainy',        # Yagmurlu gün mü? (0/1)
    'is_freezing',     # Donma noktasi alti mi? (0/1)
    'is_hot_day',      # Sicak gün mü? >30C (0/1)
    'is_temp_anomaly', # 7 günlük ort'dan +-5C uzak mi? (0/1)
]

# Null iceren feature'lar — medyan ile doldurulacak
IMPUTE_FEATURES = [
    'temp_lag_1',               # Dünkü sicaklik — en güçlü time-series feature
    'temp_rolling_avg_7d',      # 7 günlük ortalama — trend bilgisi
    'temp_diff_from_yesterday', # Dünden bugüne degisim — ani hava degisimi
]

# Kategorik feature — sayiya çevrilecek
CAT_FEATURE = 'precipitation_category'

# --- PIPELINE KURULUMU ---

# Adim 1: Null degerleri medyanla doldur
imputer = Imputer(
    inputCols=IMPUTE_FEATURES,
    outputCols=[f'{c}_imp' for c in IMPUTE_FEATURES],
    strategy='median'
)

# Adim 2: Kategorik sutunu sayiya donüstür
indexer = StringIndexer(
    inputCol=CAT_FEATURE,
    outputCol='precip_cat_idx',
    handleInvalid='keep'
)

# Adim 3: Tüm feature'lari tek vektörde topla
ALL_FEATURE_COLS = (
    STABLE_FEATURES
    + [f'{c}_imp' for c in IMPUTE_FEATURES]
    + ['precip_cat_idx']
)

assembler = VectorAssembler(
    inputCols=ALL_FEATURE_COLS,
    outputCol='features',
    handleInvalid='skip'
)

# Pipeline: adimlar sirasiya uygulanir
prep_pipeline = Pipeline(stages=[imputer, indexer, assembler])
prep_model    = prep_pipeline.fit(df)
df_prepared   = prep_model.transform(df).select('features', TARGET).na.drop()

print(f'Hazirlanan veri: {df_prepared.count():,} satir')
print(f'Kullanilan feature sayisi: {len(ALL_FEATURE_COLS)}')
print('\nFeature listesi:')
for i, feat in enumerate(ALL_FEATURE_COLS, 1):
    print(f'  {i:2d}. {feat}')

Hazirlanan veri: 10,000 satir
Kullanilan feature sayisi: 13

Feature listesi:
   1. month
   2. year
   3. day_of_year
   4. season_numeric
   5. is_weekend
   6. is_rainy
   7. is_freezing
   8. is_hot_day
   9. is_temp_anomaly
  10. temp_lag_1_imp
  11. temp_rolling_avg_7d_imp
  12. temp_diff_from_yesterday_imp
  13. precip_cat_idx


In [5]:
train_df, test_df = df_prepared.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count  = test_df.count()

print(f'Egitim seti : {train_count:,} satir (%80)')
print(f'Test seti   : {test_count:,} satir (%20)')
print(f'\nHedef degisken : {TARGET}')
print('Problem tipi   : REGRESYON (sürekli deger tahmini)')
print('\nDegerlendirme metrikleri:')
print('  RMSE — Root Mean Squared Error  (düsük = iyi)')
print('  MAE  — Mean Absolute Error      (düsük = iyi)')
print('  R2   — Açiklanan varyans orani  (1.0   = mükemmel)')

Egitim seti : 8,079 satir (%80)
Test seti   : 1,921 satir (%20)

Hedef degisken : avg_temp_c
Problem tipi   : REGRESYON (sürekli deger tahmini)

Degerlendirme metrikleri:
  RMSE — Root Mean Squared Error  (düsük = iyi)
  MAE  — Mean Absolute Error      (düsük = iyi)
  R2   — Açiklanan varyans orani  (1.0   = mükemmel)


In [6]:
evaluator_rmse = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction', metricName='rmse')
evaluator_mae  = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction', metricName='mae')
evaluator_r2   = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction', metricName='r2')

def evaluate_model(predictions):
    rmse = round(evaluator_rmse.evaluate(predictions), 4)
    mae  = round(evaluator_mae.evaluate(predictions),  4)
    r2   = round(evaluator_r2.evaluate(predictions),   4)
    return rmse, mae, r2

results        = {}   # {model_adi: {RMSE, MAE, R2}}
trained_models = {}   # {model_adi: (model_obj, predictions_df)}

print('✅ Degerlendirici hazir, model egitimi baslıyor...')

✅ Degerlendirici hazir, model egitimi baslıyor...


## Model Eğitimi

Her model için:
1. Modeli **eğitim verisi** üzerinde eğit
2. **Test verisi** üzerinde tahmin yap
3. RMSE / MAE / R² hesapla
4. **MLflow**'a parametre, metrik ve modeli kaydet

In [7]:
print('=' * 55)
print('  MODEL 1: Linear Regression')
print('=' * 55)
print('Ne yapar: y = w1*x1 + w2*x2 + ... + b formülüyle tahmin.')
print('           Basit ama yorumlanabilir bir baz model.\n')

with mlflow.start_run(run_name='LinearRegression'):
    params = {'maxIter': 100, 'regParam': 0.1, 'elasticNetParam': 0.0}

    lr = LinearRegression(
        featuresCol='features', labelCol=TARGET,
        maxIter=params['maxIter'],
        regParam=params['regParam'],
        elasticNetParam=params['elasticNetParam']
    )

    lr_model = lr.fit(train_df)
    preds    = lr_model.transform(test_df)
    rmse, mae, r2 = evaluate_model(preds)

    mlflow.log_params(params)
    mlflow.log_metrics({'RMSE': rmse, 'MAE': mae, 'R2': r2})
    mlflow.spark.log_model(lr_model, 'model')

    results['Linear Regression']        = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    trained_models['Linear Regression'] = (lr_model, preds)

print(f'✅ RMSE : {rmse:.4f} C')
print(f'   MAE  : {mae:.4f} C')
print(f'   R2   : {r2:.4f}  (%{r2*100:.1f} varyans aciklandi)')

  MODEL 1: Linear Regression
Ne yapar: y = w1*x1 + w2*x2 + ... + b formülüyle tahmin.
           Basit ama yorumlanabilir bir baz model.

✅ RMSE : 0.3333 C
   MAE  : 0.2467 C
   R2   : 0.9987  (%99.9 varyans aciklandi)


In [8]:
print('=' * 55)
print('  MODEL 2: Decision Tree Regressor')
print('=' * 55)
print('Ne yapar: Veriyi özellik esiklerine göre bölerek dal')
print('           yapar. Her yaprak bir tahmin degeridir.\n')

with mlflow.start_run(run_name='DecisionTree'):
    params = {'maxDepth': 8, 'minInstancesPerNode': 5}

    dt = DecisionTreeRegressor(
        featuresCol='features', labelCol=TARGET,
        maxDepth=params['maxDepth'],
        minInstancesPerNode=params['minInstancesPerNode']
    )

    dt_model = dt.fit(train_df)
    preds    = dt_model.transform(test_df)
    rmse, mae, r2 = evaluate_model(preds)

    mlflow.log_params(params)
    mlflow.log_metrics({'RMSE': rmse, 'MAE': mae, 'R2': r2})
    mlflow.spark.log_model(dt_model, 'model')

    results['Decision Tree']        = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    trained_models['Decision Tree'] = (dt_model, preds)

print(f'✅ RMSE : {rmse:.4f} C')
print(f'   MAE  : {mae:.4f} C')
print(f'   R2   : {r2:.4f}  (%{r2*100:.1f} varyans aciklandi)')

  MODEL 2: Decision Tree Regressor
Ne yapar: Veriyi özellik esiklerine göre bölerek dal
           yapar. Her yaprak bir tahmin degeridir.

✅ RMSE : 0.8463 C
   MAE  : 0.4723 C
   R2   : 0.9918  (%99.2 varyans aciklandi)


In [9]:
print('=' * 55)
print('  MODEL 3: Random Forest Regressor')
print('=' * 55)
print('Ne yapar: 100 farkli karar agaci egitir, tahminlerin')
print('           ortalamasini alir. Ensemble yöntemi.\n')

with mlflow.start_run(run_name='RandomForest'):
    params = {'numTrees': 100, 'maxDepth': 10, 'featureSubsetStrategy': 'auto'}

    rf = RandomForestRegressor(
        featuresCol='features', labelCol=TARGET,
        numTrees=params['numTrees'],
        maxDepth=params['maxDepth'],
        featureSubsetStrategy=params['featureSubsetStrategy'],
        seed=42
    )

    rf_model = rf.fit(train_df)
    preds    = rf_model.transform(test_df)
    rmse, mae, r2 = evaluate_model(preds)

    mlflow.log_params(params)
    mlflow.log_metrics({'RMSE': rmse, 'MAE': mae, 'R2': r2})
    mlflow.spark.log_model(rf_model, 'model')

    results['Random Forest']        = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    trained_models['Random Forest'] = (rf_model, preds)

print(f'✅ RMSE : {rmse:.4f} C')
print(f'   MAE  : {mae:.4f} C')
print(f'   R2   : {r2:.4f}  (%{r2*100:.1f} varyans aciklandi)')

  MODEL 3: Random Forest Regressor
Ne yapar: 100 farkli karar agaci egitir, tahminlerin
           ortalamasini alir. Ensemble yöntemi.

✅ RMSE : 0.8176 C
   MAE  : 0.4832 C
   R2   : 0.9923  (%99.2 varyans aciklandi)


In [10]:
print('=' * 55)
print('  MODEL 4: Gradient Boosted Trees (GBT) Regressor')
print('=' * 55)
print('Ne yapar: Her yeni agaç önceki agacin hatasini düzeltir.')
print('           Ardisik ögrenme — genellikle en güçlü yöntem.\n')

with mlflow.start_run(run_name='GBTRegressor'):
    params = {'maxIter': 100, 'maxDepth': 6, 'stepSize': 0.1}

    gbt = GBTRegressor(
        featuresCol='features', labelCol=TARGET,
        maxIter=params['maxIter'],
        maxDepth=params['maxDepth'],
        stepSize=params['stepSize'],
        seed=42
    )

    gbt_model = gbt.fit(train_df)
    preds     = gbt_model.transform(test_df)
    rmse, mae, r2 = evaluate_model(preds)

    mlflow.log_params(params)
    mlflow.log_metrics({'RMSE': rmse, 'MAE': mae, 'R2': r2})
    mlflow.spark.log_model(gbt_model, 'model')

    results['GBT']        = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    trained_models['GBT'] = (gbt_model, preds)

print(f'✅ RMSE : {rmse:.4f} C')
print(f'   MAE  : {mae:.4f} C')
print(f'   R2   : {r2:.4f}  (%{r2*100:.1f} varyans aciklandi)')

  MODEL 4: Gradient Boosted Trees (GBT) Regressor
Ne yapar: Her yeni agaç önceki agacin hatasini düzeltir.
           Ardisik ögrenme — genellikle en güçlü yöntem.

✅ RMSE : 0.7309 C
   MAE  : 0.3895 C
   R2   : 0.9939  (%99.4 varyans aciklandi)


In [11]:
print('=' * 55)
print('  MODEL 5: Generalized Linear Regression (GLR)')
print('=' * 55)
print('Ne yapar: Istatistiksel regresyon modeli. Gaussian ailesi')
print('           ile LR benzeri; detayli istatistik çiktisi verir.\n')

with mlflow.start_run(run_name='GeneralizedLinearRegression'):
    params = {'family': 'gaussian', 'link': 'identity', 'maxIter': 100, 'regParam': 0.1}

    glr = GeneralizedLinearRegression(
        featuresCol='features', labelCol=TARGET,
        family=params['family'],
        link=params['link'],
        maxIter=params['maxIter'],
        regParam=params['regParam']
    )

    glr_model = glr.fit(train_df)
    preds     = glr_model.transform(test_df)
    rmse, mae, r2 = evaluate_model(preds)

    mlflow.log_params(params)
    mlflow.log_metrics({'RMSE': rmse, 'MAE': mae, 'R2': r2})
    mlflow.spark.log_model(glr_model, 'model')

    results['GLR']        = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    trained_models['GLR'] = (glr_model, preds)

print(f'✅ RMSE : {rmse:.4f} C')
print(f'   MAE  : {mae:.4f} C')
print(f'   R2   : {r2:.4f}  (%{r2*100:.1f} varyans aciklandi)')
print('\nGLR Katsayilari (ilk 5):')
coef_df = pd.DataFrame({
    'Feature': ALL_FEATURE_COLS,
    'Katsayi': list(glr_model.coefficients)
}).sort_values('Katsayi', key=abs, ascending=False)
print(coef_df.head().to_string(index=False))

  MODEL 5: Generalized Linear Regression (GLR)
Ne yapar: Istatistiksel regresyon modeli. Gaussian ailesi
           ile LR benzeri; detayli istatistik çiktisi verir.

✅ RMSE : 0.3333 C
   MAE  : 0.2467 C
   R2   : 0.9987  (%99.9 varyans aciklandi)

GLR Katsayilari (ilk 5):
                     Feature   Katsayi
temp_diff_from_yesterday_imp  0.924427
              temp_lag_1_imp  0.812656
                 is_freezing -0.638017
                  is_hot_day  0.304911
     temp_rolling_avg_7d_imp  0.178072


## Sonuçlar ve Karşılaştırma

In [12]:
print('=' * 65)
print('  5 MODEL KARSILASTIRMA SONUÇLARI')
print('=' * 65)

results_pd = (
    pd.DataFrame(results)
    .T
    .reset_index()
    .rename(columns={'index': 'Model'})
)
results_pd = results_pd.sort_values('RMSE').reset_index(drop=True)
results_pd.insert(0, 'Siralama', range(1, len(results_pd) + 1))
results_pd['R2_pct'] = (results_pd['R2'] * 100).round(1)

print(results_pd[['Siralama', 'Model', 'RMSE', 'MAE', 'R2', 'R2_pct']].to_string(index=False))

best_model_name = results_pd.iloc[0]['Model']
print(f'\n🏆 En iyi model  : {best_model_name}')
print(f'   RMSE         : {results_pd.iloc[0]["RMSE"]:.4f} C')
print(f'   MAE          : {results_pd.iloc[0]["MAE"]:.4f} C')
print(f'   R2           : {results_pd.iloc[0]["R2"]:.4f}')

# Sonuclari Delta Lake Gold katmanina kaydet
RESULTS_PATH = '/home/jovyan/work/delta-lake/gold/model_results'
spark.createDataFrame(results_pd).write.format('delta').mode('overwrite').save(RESULTS_PATH)
print(f'\n✅ Sonuclar Delta Lake\'e kaydedildi: {RESULTS_PATH}')

  5 MODEL KARSILASTIRMA SONUÇLARI
 Siralama             Model   RMSE    MAE     R2  R2_pct
        1 Linear Regression 0.3333 0.2467 0.9987    99.9
        2               GLR 0.3333 0.2467 0.9987    99.9
        3               GBT 0.7309 0.3895 0.9939    99.4
        4     Random Forest 0.8176 0.4832 0.9923    99.2
        5     Decision Tree 0.8463 0.4723 0.9918    99.2

🏆 En iyi model  : Linear Regression
   RMSE         : 0.3333 C
   MAE          : 0.2467 C
   R2           : 0.9987

✅ Sonuclar Delta Lake'e kaydedildi: /home/jovyan/work/delta-lake/gold/model_results


In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Agaç tabanli modellerin feature importance degerleri
tree_models = {
    'Random Forest': trained_models['Random Forest'][0],
    'GBT':           trained_models['GBT'][0],
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Importance Analizi', fontsize=15, fontweight='bold')

for ax, (model_name, model_obj) in zip(axes, tree_models.items()):
    importances = model_obj.featureImportances.toArray()
    indices     = np.argsort(importances)   # küçükten büyüge (yatay bar)
    colors      = ['#1565C0' if importances[i] >= 0.05 else '#90CAF9' for i in indices]

    ax.barh(
        [ALL_FEATURE_COLS[i] for i in indices],
        [importances[i] for i in indices],
        color=colors, alpha=0.85, edgecolor='white'
    )
    ax.set_title(f'{model_name} Feature Importance', fontsize=12, fontweight='bold')
    ax.set_xlabel('Önem Skoru')
    ax.axvline(x=0.05, color='red', linestyle='--', alpha=0.6, label='Esik: 0.05')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/home/jovyan/work/notebooks/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance grafigi kaydedildi: notebooks/feature_importance.png')

# En önemli 3 feature'i yazdir
rf_imp = trained_models['Random Forest'][0].featureImportances.toArray()
top3_idx = np.argsort(rf_imp)[::-1][:3]
print('\nRandom Forest — En önemli 3 feature:')
for rank, idx in enumerate(top3_idx, 1):
    print(f'  {rank}. {ALL_FEATURE_COLS[idx]}: {rf_imp[idx]:.4f}')

✅ Feature importance grafigi kaydedildi: notebooks/feature_importance.png

Random Forest — En önemli 3 feature:
  1. temp_lag_1_imp: 0.4734
  2. temp_rolling_avg_7d_imp: 0.3037
  3. season_numeric: 0.0712


In [14]:
best_model_obj, best_preds = trained_models[best_model_name]
pred_pd = best_preds.select(TARGET, 'prediction').toPandas()
pred_pd['residual'] = pred_pd[TARGET] - pred_pd['prediction']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'En Iyi Model Analizi: {best_model_name}', fontsize=14, fontweight='bold')

# --- Scatter: Gercek vs Tahmin ---
lim_min = min(pred_pd[TARGET].min(), pred_pd['prediction'].min()) - 1
lim_max = max(pred_pd[TARGET].max(), pred_pd['prediction'].max()) + 1

axes[0].scatter(pred_pd[TARGET], pred_pd['prediction'],
                alpha=0.4, s=20, color='steelblue', label='Tahminler')
axes[0].plot([lim_min, lim_max], [lim_min, lim_max],
             'r--', linewidth=2, label='Mükemmel Tahmin')
axes[0].set_xlabel('Gercek Sicaklik (C)', fontsize=11)
axes[0].set_ylabel('Tahmin Edilen Sicaklik (C)', fontsize=11)
axes[0].set_title('Gercek vs Tahmin')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# R2 degerini grafige ekle
r2_val = results[best_model_name]['R2']
axes[0].text(0.05, 0.92, f'R² = {r2_val:.4f}',
             transform=axes[0].transAxes, fontsize=12,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Residual Histogram ---
res_mean = pred_pd['residual'].mean()
res_std  = pred_pd['residual'].std()

axes[1].hist(pred_pd['residual'], bins=40, color='salmon', alpha=0.8, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Sifir Artik')
axes[1].axvline(x=res_mean, color='navy', linestyle='-.', linewidth=2,
                label=f'Ort: {res_mean:.2f} C')
axes[1].set_xlabel('Artik (Residual = Gercek - Tahmin) C', fontsize=11)
axes[1].set_ylabel('Frekans')
axes[1].set_title('Residual Dagilimi')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/home/jovyan/work/notebooks/best_model_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n--- Residual Istatistikleri ({best_model_name}) ---')
print(f'Ortalama artik : {res_mean:.4f} C  (sifira yakinsa model yanli degil)')
print(f'Std sapma      : {res_std:.4f} C')
print(f'Max abs artik  : {pred_pd["residual"].abs().max():.4f} C')
print('✅ Grafik kaydedildi: notebooks/best_model_analysis.png')


--- Residual Istatistikleri (Linear Regression) ---
Ortalama artik : -0.0025 C  (sifira yakinsa model yanli degil)
Std sapma      : 0.3334 C
Max abs artik  : 2.0463 C
✅ Grafik kaydedildi: notebooks/best_model_analysis.png


In [15]:
print('=' * 65)
print('  MLFLOW DENEY ÖZETI')
print('=' * 65)
print('Tracking URI : file:///home/jovyan/work/mlflow')
print('Deney adi    : climate_temperature_regression')
print(f'Toplam run   : {len(results)}')
print()
print('MLflow UI icin terminalde:')
print('  mlflow ui --backend-store-uri /home/jovyan/work/mlflow --port 5000')
print('  Tarayicida : http://localhost:5000')
print()
print('=' * 65)
print('  KAYDEDILEN DOSYALAR')
print('=' * 65)
print('  notebooks/feature_importance.png')
print('  notebooks/best_model_analysis.png')
print('  delta-lake/gold/model_results  (Delta tablosu)')
print()
print('=' * 65)
print('  ADIM 6 TAMAMLANDI ✅')
print('=' * 65)

spark.stop()
print('Spark Session kapatildi.')

  MLFLOW DENEY ÖZETI
Tracking URI : file:///home/jovyan/work/mlflow
Deney adi    : climate_temperature_regression
Toplam run   : 5

MLflow UI icin terminalde:
  mlflow ui --backend-store-uri /home/jovyan/work/mlflow --port 5000
  Tarayicida : http://localhost:5000

  KAYDEDILEN DOSYALAR
  notebooks/feature_importance.png
  notebooks/best_model_analysis.png
  delta-lake/gold/model_results  (Delta tablosu)

  ADIM 6 TAMAMLANDI ✅
Spark Session kapatildi.
